# Module 3 — NGS Pipeline with Claude Code (Alignment)

**SBML Lab Intern Training | KAIST**

---

This module covers the full NGS alignment pipeline — from raw SRA data to a sorted, indexed BAM, a GFF file, and a visualization of the ChIP-exo signal in MetaScope. You'll use Claude Code to generate the pipeline commands, but you must understand what each step does before running it.

**Lab context:** Reference genome is *E. coli* K-12 MG1655. The dataset is **single-end ChIP-exo reads** — each read comes from one end of the DNA fragment, not a pair. Aligner is `bowtie2`. Post-alignment processing uses `samtools`, followed by `makegff.py` to produce a GFF file for MetaScope visualization.

**Learning objectives:**
- Understand how `bowtie2` aligns short reads to a reference genome
- Understand why BAM files must be sorted and indexed before downstream use
- Practice using Claude Code's **plan mode** for multi-step pipelines
- Annotate and verify every flag in a generated command before running it

## Background: NGS and ChIP-exo

### What is next-generation sequencing (NGS)?

A sequencing machine reads millions of short DNA fragments (called **reads**) in parallel and reports their sequences along with quality scores for each base. A single ChIP-exo experiment produces 10–50 million reads, each 50–150 base pairs long. Raw reads are stored in **FASTQ** format — one file, millions of entries.

Reads alone don't tell you anything. You need to figure out *where* in the genome each read came from. That's what alignment does.

### What is ChIP-exo?

**ChIP-exo** is a technique that maps exactly where a DNA-binding protein (a transcription factor) attaches to the chromosome:

1. **ChIP (chromatin immunoprecipitation):** Protein-DNA complexes are chemically cross-linked and then broken into fragments. An antibody specific to your protein of interest pulls down only the fragments it was bound to.
2. **Exo (exonuclease treatment):** A DNA-chewing enzyme trims the pulled-down fragments from their ends, leaving only the sequence right at the binding boundary. This is what makes ChIP-*exo* more precise than regular ChIP-seq.
3. **Sequencing:** The remaining fragments are sequenced. The resulting reads cluster tightly around the exact binding site.

### Why does this lab do ChIP-exo?

This lab studies transcription factors (TFs) — proteins that bind specific DNA sequences to control nearby genes. ChIP-exo lets us answer: *"Where in the E. coli genome does TF X bind, and under what conditions?"*

The dataset you'll work with in Modules 3 and 4 is a FUR ChIP-exo experiment — mapping where the iron-sensing protein FUR binds across the entire *E. coli* chromosome.

### The pipeline in one sentence

`FASTQ reads` → **align to reference genome** → `SAM/BAM alignment file` → **sort and index** → **convert to GFF** → `visualize in MetaScope`

Each step in this pipeline is one section of this module.

---

## 0. File Formats — Know Before You Align

This pipeline produces and consumes several file formats. Use Claude Code to answer the questions below **before** starting the exercises.

| Format | Question to answer with Claude Code |
|--------|-------------------------------------|
| FASTQ  | What do the 4 lines per read represent? What does the quality score number mean? |
| FASTA  | How does it differ from FASTQ? When would you use one vs the other? |
| SAM    | How many columns are there? What does the FLAG field encode? |
| BAM    | Why is BAM preferred over SAM for storage and downstream tools? |
| GFF    | How many columns? What coordinate system (0-based or 1-based)? |

Write your answers below — one sentence per format.


> **Answer**
>
> FASTQ: 4 줄: (1) @ + read ID/description (2) the raw nucleotide sequence (3) a + (optionally repeating the ID) (4) quality score, a string of ASCII characters the same length as the sequence, 각 문자는 해당 위치의 염기의 quality를 encode한다. Quality는 Phred score: Q = -10*log10(P), 여기서 P = 해당 염기(base call)가 틀렸을 확률. Q30이면 -> base call이 틀렸을 확률은 1/1000 (99.9% accurate). 대부분의 Illumina data는 Phred+33 encoding을 사용 -> ! = Q0, I = Q40
> FASTA: FASTA는 entry 별로 2줄 (>header, sequence)만 있다. FASTQ(error-prone)와 다르게 이미 알려져 있거나 확정된 서열 정보를 담는다. 시퀀싱을 통해 나오는 정보(ChIP-exo & RNA-seq reads)는 alignment 이전 트리밍/필터링을 거쳐야 하기 때문에 FASTQ를 사용한다. 이미 확정(settled)되어 염기 별 confidence 정보가 필요하지 않은 서열(E. coli K-12 MG1655 reference genome, MEME Suite motif hits, Biopython-extracted sequences)은 FASTA를 사용한다.
> SAM: 11개 column: (1) QNAME read name (2) FLAG bitwise status (3) RNAME reference name (4) POS 1-based leftmost mapping position (5) MAPQ mapping quality (6) CIGAR alignment string (matches/indels/clips) (7) RNEXT reference of the mate (single-end는 *) (8) PNEXT mate's position (single-end는 0) (9) TLEN template length (single-end는 0) (10) SEQ the read sequence (11) QUAL the read's quality string. FLAG는 독립된 특성들에 대한 boolean 플래그를 하나의 bitwise 정수로 압축한 것이다. 예: 99 = 64+32+2+1 -> first in pair, mate reverse strand, read mapped in proper pair, read paired
> BAM: SAM의 압축된, 이진법 encoding된 형식. BAM은 BGZF 압축기법을 사용하여, 파일 크기를 대략 4-10배 줄인다. 또한 좌표에 따라 정렬하고 .bai index 파일을 생성하면 파일 전체를 압축해제하지 않고 특정 genomic region을 바로 사용할 수 있다. 반면, SAM 파일은 인덱스가 없기 때문에 도구 사용할 때마다 처음부터 순서대로 읽게 된다. 따라서 파일 저장 및 다운스트림 활용 시에 BAM 파일이 선호된다.
> GFF: 9개 column; (1) seqid - reference sequence name (2) source - tool/database that generated the feature (3) type - feature type (gene, CDS, exon...) (4) start pos (5) end pos (6) score (7) strand (+/-/.) (8) phase (CDS의 경우 reading frame offset) (9) attributes - semicolon-separated key=value pairs. 좌표는 1-based, fully closed(start부터 end까지 모두 포함됨)


## 1. How `bowtie2` Works

### What is alignment?

**Alignment** is the process of finding where each sequenced read comes from in the reference genome. Imagine you have 20 million short sentences (reads), each 75 words long, and you want to find where each one appears in a 4.6-million-word book (the *E. coli* genome). Alignment does that — at scale, in seconds.

The output is a file where every read is annotated with its genomic position: chromosome, start coordinate, strand, and a mapping quality score.

### Why do we need an index?

Searching the entire genome for every read would be impossibly slow. `bowtie2-build` pre-processes the reference genome into a compressed **index** — a data structure that allows near-instant lookup. Think of it like a book index: instead of reading every page to find "FUR binding site," you jump straight to the relevant pages.

You build the index once; then alignment against that genome uses it every time.

`bowtie2` aligns short reads to a reference genome using an index built with `bowtie2-build`. Use Claude Code to understand how the aligner works before you run it.

### Exercise 1 — Understand bowtie2 Before Running It

Use Claude Code to understand how `bowtie2` finds alignments and what the key flags control. Write a 3-sentence summary in your own words in the cell below.

> **Answer**
>
> bowtie2-build가 먼저 reference genome의 FM-index를 생성한다.
> bowtie2가 각 리드에서 짧은 seed substring들을 추출하여, FM-index에서 후보 매칭 위치를 찾는다. 
> 후보 hit 주변에서 Smith-Waterman 계열 dynamic programming으로 gapped alignment를 확장하고, score 기준을 만족하는 end-to-end(read 전체) 또는 local alignment(리드 양 끝 일부 soft-clipping)를 보고한다.


## 2. `samtools` Internals

### Why sort a BAM file?

After alignment, reads are stored in the order they were sequenced — essentially random with respect to genomic position. Most downstream tools (genome browsers, variant callers, coverage calculators) need to jump to a specific genomic region quickly. A **coordinate-sorted BAM** organizes reads by position, so a tool looking at position 1,234,567 doesn't have to scan the whole file — it jumps directly there.

**You must sort before you index.** An index built on an unsorted file is meaningless because the file's structure doesn't match what the index promises.

### Why index a BAM file?

The `.bai` index file is a lookup table that maps genomic positions to byte offsets in the BAM file. It's what allows `samtools view` to extract reads from chromosome position X in milliseconds. Without it, every query would require reading the entire BAM from the start.

`samtools` converts, sorts, and indexes alignment files. Use Claude Code to understand the difference between SAM and BAM, and why the sort step must come before the index step.

### Exercise 2 — Sort Before Index

**Without asking Claude Code first:** write in the cell below why you must sort a BAM file before indexing it. Use your own reasoning.

Then verify your explanation with Claude Code.

> **Answer**
>
> 인덱싱을 하는 것은 나중에 파일을 열 때, BAM에서 한번에 특정 위치를 열고 사용하기 위해서인데, 인덱싱하기 전 sorting을 하지 않으면, 특정 위치 주변의 리드들이 파일 전체에 흩어져 있어 이들을 빠르게 찾기 어렵게 된다.

## 3. Plan Mode — Reviewing Pipelines Before Running

Claude Code's **plan mode** lets you see every step Claude intends to take before a single command is executed.

**How to activate:** Press **Shift+Tab** before sending your prompt, cycling until the status line shows plan mode is on (a single press may land on a different mode). The interface switches to plan mode. Claude will show you the full sequence of actions — file reads, commands, edits — and wait for your approval.

**Why this matters for pipelines:**  
A multi-step NGS pipeline touches raw data, builds index files, and writes intermediate outputs. A mistake at step 2 (wrong index path) silently produces a valid-looking but incorrect BAM. Reviewing the plan lets you:
- Catch wrong paths or flag values before they cause silent errors
- Understand every command before it runs
- Modify individual steps (e.g., add `--no-unal`, increase `-p` threads) without re-generating the whole plan

**Rule for this module:** Any time you ask Claude Code to generate a multi-step pipeline, you must use plan mode and review it before approving.

### Exercise 3 — Plan Mode Pipeline Review

Use plan mode (Shift+Tab twice) to generate the full pipeline:
```
FASTQ → bowtie2 align → samtools sort → samtools index → makegff.py
```
Review every proposed step before approving. Identify at least one flag or parameter you would change or verify, and explain why.

> **Answer**
>
> *(Write the flag/parameter you identified, your change, and your reasoning here.)*


## 4. Downloading Your Input Data

The pipeline needs **two** inputs, both in `data/reference/`: the sequencing **reads** and the **reference genome** they align to. You'll download both with Claude Code in this section.

### Exercise 4 — Reads: single-end ChIP-exo (`fastq-dump`)

The reads come from the FUR ChIP-exo study you'll work with in Module 4: Seo et al. 2014, GEO series **GSE54901**. Use Claude Code to find and download an appropriate single-end ChIP-exo sample into `data/reference/`.

In [ ]:
%%bash
# Exercise 4 — Use Claude Code to download the ChIP-exo reads from GSE54901.
# Paste the final command(s) here and annotate each flag with a comment.


### Exercise 4b — Reference Genome: *E. coli* K-12 MG1655 (`NC_000913.3`)

Alignment needs the genome the reads came from. Use Claude Code to download the *E. coli* K-12 MG1655 reference genome — accession **NC_000913.3** — as a FASTA into `data/reference/`, saved as `NC_000913.3.fasta`. (Hint: the `entrez-direct` tools in this container can fetch sequences by accession from NCBI.)

This is the file `bowtie2-build` turns into the alignment index in Exercise 7 — without it the pipeline's first step fails.

In [ ]:
%%bash
# Exercise 4b — Use Claude Code to download the reference genome NC_000913.3 as FASTA
#               into data/reference/NC_000913.3.fasta.
# Paste the final command here and annotate what it does.


## 5. SAM FLAG Values

Every SAM record has a FLAG field in column 2 — a bitwise integer encoding alignment properties. Use Claude Code to understand what FLAG values mean and how to filter by them with `samtools view`.

### Exercise 5 — Decode an Unknown FLAG

> Look up what a specific FLAG value means using Claude Code. Choose a value you have **not** looked up yet (e.g., 2048, 256, 1024).
>
> Write:
> 1. The FLAG value you chose
> 2. What it means (decoded bits)
> 3. How you would verify that a real SAM record carries this flag (e.g., with `samtools view -f` or `samtools flags`)

> **Answer**
>
> *(Write your FLAG value, its meaning, and verification method here.)*


## 6. Writing `makegff.py`

### What does makegff.py do and why do we need it?

Your sorted BAM file contains alignment data in a binary format that bioinformatics tools understand — but **MetaScope**, the lab's genome browser, reads GFF files, not BAM.

`makegff.py` is a conversion script: it reads each aligned read from the BAM and writes one GFF row for it. The resulting GFF file can be loaded into MetaScope alongside the gene annotation from Module 2, letting you visually inspect where ChIP-exo reads piled up relative to annotated genes.

This script is **not pre-provided** — you will write it using Claude Code. This is intentional: `makegff.py` is a simple script, but writing it forces you to understand both the BAM format (what pysam gives you) and the GFF format (what MetaScope expects). Consult `lab-context.md` for the exact GFF column structure this lab uses.

`makegff.py` is the final step in the pipeline — it converts your sorted BAM into a GFF file that MetaScope can load. Use Claude Code to write it before running the full pipeline.

### Exercise 6 — Write `makegff.py`

Use Claude Code to write a Python script named `makegff.py` that:
1. Takes a sorted BAM file as input
2. Outputs a GFF-format file readable by MetaScope

Test it on your aligned BAM before running the full pipeline in Exercise 7.

Write the final working script in the code cell below.

**Expected output format** (one row per aligned read):
```
NC_000913.3	makegff	read	12345	12396	1	+	.	name=SRR000000.1
```
Columns: seqname, source (`makegff`), feature (`read`), start, end, score (value: `1` per read), strand, `.`, read name.
Coordinates are **1-based**. Each aligned read becomes one GFF row.


> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
# Write your makegff.py here.
# IMPORTANT: this script must be SAVED as a file (notebooks/makegff.py), because
# Exercise 7 runs it from the terminal as:  python notebooks/makegff.py <bam> <gff>
# Easiest way from a notebook: make the FIRST line of this cell
#     %%writefile notebooks/makegff.py
# so running the cell writes the script to disk instead of executing it here.
# (If you run the script body directly in this cell, sys.argv will be the Jupyter
#  kernel's arguments, not your BAM path, and it will fail.)


## 7. Running the Full Pipeline

### Exercise 7 — Run the Full Pipeline

By now `data/reference/` should hold both inputs from Section 4: the reference genome (`NC_000913.3.fasta`, Exercise 4b) and the ChIP-exo reads (Exercise 4). Use plan mode to generate the full pipeline, then fill in each command below after reviewing the plan.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [ ]:
%%bash
# Full NGS pipeline — fill in the commands after generating them with Claude Code in plan mode.
# Do NOT run this cell until you have reviewed the plan and annotated each flag.

# Step 1: Build bowtie2 index from reference FASTA


# Step 2: Align single-end reads with bowtie2


# Step 3: Convert SAM to BAM (samtools view)


# Step 4: Sort BAM by coordinate (samtools sort)


# Step 5: Index sorted BAM (samtools index)


# Step 6: Run your makegff.py to generate GFF for MetaScope


## 8. Visualizing ChIP-exo in MetaScope

You now have a GFF file of aligned ChIP-exo reads. Numbers in a file don't tell you where FUR binds — you need to *see* the reads piled up along the genome next to the genes they sit near. That's what a **genome browser** does.

### What is a genome browser?

A genome browser draws the genome as a horizontal axis (position 1 → 4.6 Mb for *E. coli*) and stacks your data on top as **tracks**. Each track is one GFF file. Load the gene annotation as one track and your ChIP-exo reads as another, and you can read directly off the screen: *this pile of reads sits right in front of that gene.*

### MetaScope — the lab's browser

**MetaScope** is the SBML lab's own genome browser. It reads **GFF files** (exactly what `makegff.py` produced) and lets you overlay multiple tracks, zoom, jump to a position, search by gene name, and export a publication-quality figure.

- **It is a desktop application** — it runs on your own computer (Windows or macOS), **not** inside the Codespace. The Codespace is where you *make* the GFF; MetaScope is where you *look* at it.
- **Download and install it from the lab homepage** ([sbml-lab.ai](https://sbml-lab.ai)). Pick the build for your operating system.
- The workflow is: **produce the GFF in the Codespace → download the GFF to your machine → open it in MetaScope → cross-reference with the annotation → export a figure.**

### The two tracks you'll load

| Track | File | What it shows |
|-------|------|---------------|
| Gene annotation | `ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (from Module 2) | Where every gene is |
| ChIP-exo reads | your `makegff.py` output (Exercise 6/7) | Where FUR-bound DNA fragments piled up |

Overlaid, a tall stack of ChIP-exo reads sitting just upstream of a gene is a candidate **FUR binding site** regulating that gene.

### Exercise 8 — What Should a Binding Site Look Like? (write before you look)

**Before installing anything**, write your prediction in the cell below — reason it out yourself:

1. ChIP-exo enriches DNA fragments that were bound by the FUR protein. When those reads are aligned and drawn as a track, **what shape** would you expect at a real binding site versus a random stretch of genome?
2. FUR is a **repressor** of iron-uptake genes. Relative to a gene it controls, **where** along the genome would you expect its binding site to sit — inside the gene, or somewhere else?

Then verify your reasoning with Claude Code (`/explain ChIP-exo peak shape` or ask directly).

> **Answer**
>
> *(Write your two predictions here, before opening MetaScope.)*

### Exercise 9 — Get Your GFFs Out of the Codespace

MetaScope runs on your machine, so both tracks need to be on your machine. Run the cell below to confirm both GFFs exist and see their sizes, then download them.

**To download a file from a Codespace:** in the VS Code file explorer (left panel), right-click the file → **Download…**, and save it somewhere you'll find it (e.g. your Desktop).

Download **both**:
- `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (the annotation)
- your ChIP-exo GFF from Exercise 6/7 (e.g. `notebooks/chipexo.gff` — use whatever name you gave it)

In [ ]:
%%bash
# Confirm both tracks exist and are non-empty before you download them.
# Adjust the ChIP-exo filename to whatever you named your makegff.py output.

echo "== Annotation track =="
ls -lh data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
head -2 data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff

echo
echo "== ChIP-exo track (edit the filename if yours differs) =="
CHIP_GFF=notebooks/chipexo.gff
ls -lh "$CHIP_GFF" && head -2 "$CHIP_GFF" && echo "rows: $(wc -l < "$CHIP_GFF")"


### Exercise 10 — Load, Locate, and Identify

Open MetaScope on your machine and:

1. **Open both GFFs** (File → Open, or `Ctrl+O` / `Cmd+O`). Load the annotation *and* your ChIP-exo track.
2. **Find a binding site.** Look for a location where the ChIP-exo track shows a clear, tall pile of reads. Use `Ctrl/Cmd+G` to jump to a position and the horizontal scroll/zoom to explore. (Some well-known iron-uptake regions to check: around **160–170 kb** near *fhuA*, **609–613 kb** near the *fepA*/*fes* pair, **624–629 kb** at the enterobactin *entCEBA* genes, or **1,733 kb** near *sodB*.)
3. **Identify the nearest gene.** Hover over a gene feature to see its tooltip (position, strand, name), or use `Ctrl/Cmd+F` to search the annotation by gene name. Read off which gene the ChIP-exo pile sits **upstream of**.

> **Heads-up — a real gotcha.** MetaScope groups tracks by their **chromosome ID** (column 1 of the GFF). The annotation uses `NC_000913`, but `makegff.py` writes `NC_000913.3` (the accession of the reference FASTA). If your two tracks land on **separate chromosome tabs** and won't line up, that's why. Use `/debug` to reason it out, then fix it (make the two column-1 values match) and reload. This is exactly the kind of thing worth verifying rather than assuming.

4. **Export your figure** (`Ctrl+Shift+E` → PNG, 300 dpi). This PNG is your deliverable — save it as `module3_chipexo_metascope.png`.

In the answer cell, record: the genomic position of the site you found, the nearest gene, and whether the binding site is upstream of it (as you predicted in Exercise 8).

> **Answer**
>
> - Binding-site position (approx.): *(e.g. ~611,900)*
> - Nearest gene / locus_tag: *(e.g. fepA / b0584)*
> - Is the site upstream of the gene? Does it match your Exercise 8 prediction?
> - Attach `module3_chipexo_metascope.png` (your exported MetaScope figure) to your submission.

### Exercise 11 — Cross-Check the Biology

You found a gene next to a FUR binding site. Now confirm it makes biological sense — don't take the browser (or Claude) at face value.

Ask Claude Code: *is this gene a known FUR target, and what does it do?* A correct result should be an **iron-acquisition or iron-storage gene** (siderophore transport, enterobactin biosynthesis, iron-superoxide dismutase, etc.) — FUR represses these under iron-replete conditions.

If the gene Claude describes has nothing to do with iron, that's a signal to re-check: did you pick the right pile of reads? Is it really the *nearest* gene? Are the two tracks on the same chromosome tab? Write what you found.

> **Answer**
>
> *(What does the gene do? Is it a known FUR target? Did anything not line up, and how did you resolve it?)*

## End of Session

Before closing:

1. Make sure your pipeline cell runs end-to-end without errors.
2. Verify that the output BAM is coordinate-sorted: `samtools view -H output.bam | grep SO`
3. Verify the index exists: `ls -lh output.bam.bai`
4. Verify your GFF output has content: `wc -l output.gff`
5. Load your GFF in MetaScope and export a figure (Section 8): `module3_chipexo_metascope.png`.
6. Run `/log` in Claude Code to save a session log.

---

**Run `/log` before closing.**

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module3): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
